In [1]:
import pandas as pd
import numpy as np
# 导入必要库


In [2]:
# 读取数据并命名
orders = pd.read_csv('archive/olist_orders_dataset.csv')
items = pd.read_csv('archive/olist_order_items_dataset.csv')
customers = pd.read_csv('archive/olist_customers_dataset.csv')
products = pd.read_csv('archive/olist_products_dataset.csv')

In [3]:
# 使用for循环打印数据基础信息
for name, df in [('orders', orders), ('items', items),
                       ('customers', customers), ('products', products)]:
    print(f"=== {name} ===")
    print(f"Shape: {df.shape}")
    print(f"Missing:\n{df.isnull().sum()}")
    print(f"Dtypes:\n{df.dtypes}")
    print()

=== orders ===
Shape: (99441, 8)
Missing:
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64
Dtypes:
order_id                         object
customer_id                      object
order_status                     object
order_purchase_timestamp         object
order_approved_at                object
order_delivered_carrier_date     object
order_delivered_customer_date    object
order_estimated_delivery_date    object
dtype: object

=== items ===
Shape: (112650, 7)
Missing:
order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
dtype: int64
Dtypes:
order_id                object
order_item_id            int6

In [4]:
# 命名一个列表存储时间数据列名，使用for循环将时间数据都处理为datetime格式，方便后续对时间进行处理
date_cols = ['order_purchase_timestamp', 'order_approved_at',
             'order_delivered_carrier_date', 'order_delivered_customer_date',
             'order_estimated_delivery_date']
for col in date_cols:
    orders[col] = pd.to_datetime(orders[col])

In [5]:
# bool类型处理已交付订单，copy创建副本，后续处理要对副本进行处理，避免污染原数据，未交付订单无分析意义
# 打印已交付订单数量
orders_delivered = orders[orders['order_status'] == 'delivered'].copy()
print(f"已交付订单数: {len(orders_delivered)}")

已交付订单数: 96478


In [6]:
# 删除已交付订单中的'order_delivered_customer_date'列的缺失值  dropna
orders_delivered = orders_delivered.dropna(
    subset=['order_delivered_customer_date'])

In [7]:
# 对products文件中['product_category_name']列的缺失值填充为unkown
products['product_category_name'] = products['product_category_name'].fillna('unknown')

In [8]:
# merge合并，items文件和处理后的orders_delivered聚合，通过order_id，inner内连接（保留order_id都有的）
# 处理后得到的df文件和products[['product_id', 'product_category_name']]聚合，通过product_id，left保留左边，右边不匹配填NA
df = items.merge(orders_delivered, on='order_id', how='inner')
df = df.merge(products[['product_id', 'product_category_name']],
              on='product_id', how='left')
df = df.merge(customers[['customer_id', 'customer_unique_id', 'customer_state', 'customer_city']],
              on='customer_id', how='left')

In [9]:
# .dt的作用是：告诉pandas"我要把这个 Series 当成日期时间来操作" ，然后才能调用日期相关的方法。
# 对datetime数据各部分进行处理，以便后续使用
df['purchase_year'] = df['order_purchase_timestamp'].dt.year
df['purchase_month'] = df['order_purchase_timestamp'].dt.to_period('M')  #转成"年-月"
df['purchase_date'] = df['order_purchase_timestamp'].dt.date             #提取日期部分（去掉时分秒）
df['total_value'] = df['price'] + df['freight_value']

In [10]:
# 处理df中完全重复的行，并打印完全重复的行数
print(f"完全重复行数: {df.duplicated().sum()}")
df = df.drop_duplicates()

完全重复行数: 0


In [11]:
# 保存清洗好的文件，并检查清洗好的文件的基本信息
df.to_csv('archive/cleaned_olist.csv', index=False, encoding='utf-8-sig')
print(f"清洗完成，宽表形状: {df.shape}")
print(f"字段: {list(df.columns)}")

清洗完成，宽表形状: (110189, 22)
字段: ['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'product_category_name', 'customer_unique_id', 'customer_state', 'customer_city', 'purchase_year', 'purchase_month', 'purchase_date', 'total_value']


In [12]:
# 整体销售趋势
# groupby按月份分组，agg聚合，前为列名，后为聚合方法。groupby后分组键会变成索引，.reset_index()将之变回正常列
# .astype(str)是为了方便画图时正常显示
monthly = df.groupby('purchase_month').agg(
    order_count=('order_id', 'nunique'),
    gmv=('total_value', 'sum'),
    customer_count=('customer_unique_id', 'nunique')
).reset_index()
monthly['purchase_month'] = monthly['purchase_month'].astype(str)
monthly['aov'] = monthly['gmv'] / monthly['order_count']
monthly

,purchase_month,order_count,gmv,customer_count,aov
0,2016-09,1,143.46,1,143.460000
1,2016-10,265,46490.66,262,175.436453
2,2016-12,1,19.62,1,19.620000
3,2017-01,750,127482.37,718,169.976493
4,2017-02,1653,271239.32,1630,164.089123
5,2017-03,2546,414330.95,2508,162.738001
6,2017-04,2303,390812.40,2274,169.697091
7,2017-05,3545,566657.40,3478,159.846939
8,2017-06,3135,490050.37,3076,156.315907
9,2017-07,3872,566299.08,3802,146.254928


In [28]:
# 品类分析
# 分组/聚合/排序
category = df.groupby('product_category_name').agg(
    gmv=('total_value', 'sum'),
    order_count=('order_id', 'nunique'),
    avg_price=('price', 'mean')
).reset_index().sort_values('gmv', ascending=False)

In [14]:
# :.1%保留一位小数的百分数
top10_gmv = category.head(10)['gmv'].sum()
total_gmv = category['gmv'].sum()
print(f"Top 10 品类销售额占比: {top10_gmv/total_gmv:.1%}")

Top 10 品类销售额占比: 62.4%


In [15]:
# 地域分析
# 分组/聚合/排序
state = df.groupby('customer_state').agg(
    order_count=('order_id', 'nunique'),
    gmv=('total_value', 'sum')
).reset_index().sort_values('order_count', ascending=False)

print(f"SP（圣保罗州）订单占比: {state.iloc[0]['order_count']/state['order_count'].sum():.1%}")

SP（圣保罗州）订单占比: 42.0%


In [16]:
# 客单价分布
order_value = df.groupby('order_id')['total_value'].sum().reset_index()
order_value.columns = ['order_id', 'order_total']
print(order_value['order_total'].describe())

count    96470.000000
mean       159.825799
std        218.802007
min          9.590000
25%         61.850000
50%        105.280000
75%        176.260000
max      13664.080000
Name: order_total, dtype: float64


In [17]:
# 计算 RFM 值
# lambda函数，代入order_purchase_timestamp字段中的数据找最大值
# pd.Timedelta(days=1)数据截止日期加一天
snapshot_date = df['order_purchase_timestamp'].max() + pd.Timedelta(days=1)
rfm = df.groupby('customer_unique_id').agg(
    recency=('order_purchase_timestamp',
             lambda x: (snapshot_date - x.max()).days),
    frequency=('order_id', 'nunique'),
    monetary=('total_value', 'sum')
).reset_index()
rfm.columns = ['customer_unique_id', 'recency', 'frequency', 'monetary']

In [18]:

# qcut函数，分段q为均分几段，labels标签
# rank排序，因为有大量F数据为1，会导致出错
rfm['R_score'] = pd.qcut(rfm['recency'], q=5, labels=[5,4,3,2,1]).astype(int)
rfm['F_score'] = pd.qcut(rfm['frequency'].rank(method='first'),
                             q=5, labels=[1,2,3,4,5]).astype(int)
rfm['M_score'] = pd.qcut(rfm['monetary'], q=5, labels=[1,2,3,4,5]).astype(int)
rfm['RFM_score'] = rfm['R_score'].astype(str) + \
                       rfm['F_score'].astype(str) + \
                       rfm['M_score'].astype(str)

In [19]:

# .apply函数会将前数据按行或列代入括号内函数并运行
def rfm_segment(row):
    r, f, m = row['R_score'], row['F_score'], row['M_score']
    if r >= 4 and f >= 4 and m >= 4:
        return '重要价值用户'
    elif r >= 4 and f <= 2 and m >= 4:
        return '重要发展用户'
    elif r <= 2 and f >= 4 and m >= 4:
        return '重要保持用户'
    elif r <= 2 and f <= 2 and m >= 4:
        return '重要挽留用户'
    elif r >= 4 and f >= 4 and m <= 2:
        return '一般价值用户'
    elif r >= 4 and f <= 2 and m <= 2:
        return '一般发展用户'
    elif r <= 2 and f >= 4 and m <= 2:
        return '一般保持用户'
    else:
        return '流失用户'
rfm['segment'] = rfm.apply(rfm_segment, axis=1)

In [24]:

# 分组/聚合/排序/计算
segment_stats = rfm.groupby('segment').agg(
    user_count=('customer_unique_id', 'count'),
    total_monetary=('monetary', 'sum'),
    avg_monetary=('monetary', 'mean')
).reset_index()
segment_stats['user_pct'] = segment_stats['user_count'] / segment_stats['user_count'].sum()
segment_stats['gmv_pct'] = segment_stats['total_monetary'] / segment_stats['total_monetary'].sum()
segment_stats = segment_stats.sort_values('total_monetary', ascending=False)
segment_stats

,segment,user_count,total_monetary,avg_monetary,user_pct,gmv_pct
3,流失用户,51832,7013362.12,135.309502,0.555244,0.454870
4,重要价值用户,6498,2027091.16,311.956165,0.069609,0.131472
5,重要保持用户,5961,1861525.04,312.284019,0.063856,0.120734
6,重要发展用户,6016,1791353.32,297.764847,0.064446,0.116183
7,重要挽留用户,5726,1768609.85,308.873533,0.061339,0.114708
1,一般保持用户,5916,331074.85,55.962618,0.063374,0.021473
2,一般发展用户,5864,321515.02,54.828619,0.062817,0.020853
0,一般价值用户,5537,303863.47,54.878720,0.059314,0.019708


In [27]:
# 先拿到每个用户的首次购买月份（用 customer_unique_id）

first_order = df.groupby('customer_unique_id')['order_purchase_timestamp'].min().reset_index()
first_order.columns = ['customer_unique_id', 'first_purchase']
first_order['cohort_month'] = first_order['first_purchase'].dt.to_period('M')
# 合并回原数据
df_cohort = df.merge(first_order[['customer_unique_id', 'cohort_month']],
                     on='customer_unique_id', how='left')
# 计算每次购买属于「第几个月」（相对于首次购买）
df_cohort['order_month'] = df_cohort['order_purchase_timestamp'].dt.to_period('M')
df_cohort['months_since_first'] = (
    (df_cohort['order_month'] - df_cohort['cohort_month']).apply(lambda x: x.n)
)
# 构建同期群透视表：行=首次月，列=第N月，值=留存用户数
cohort = df_cohort.groupby(['cohort_month', 'months_since_first'])['customer_unique_id'].nunique().reset_index()
cohort_pivot = cohort.pivot(index='cohort_month', columns='months_since_first', values='customer_unique_id')
# 转换为留存率（除以第 0 月的基数）
cohort_size = cohort_pivot.iloc[:, 0]
retention = cohort_pivot.divide(cohort_size, axis=0)
retention

months_since_first,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,19,20
cohort_month,,,,,,,,,,,,,,,,,,,,
2016-09,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2016-10,1.0,NaN,NaN,NaN,NaN,NaN,0.003817,NaN,NaN,0.003817,NaN,0.003817,NaN,0.003817,NaN,0.003817,NaN,0.003817,0.007634,0.007634
2016-12,1.0,1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2017-01,1.0,0.002789,0.002789,0.001395,0.004184,0.001395,0.004184,0.001395,0.001395,NaN,0.004184,0.001395,0.006974,0.004184,0.001395,0.001395,0.002789,0.004184,0.001395,NaN
2017-02,1.0,0.001843,0.003071,0.001229,0.004300,0.001229,0.002457,0.001843,0.001229,0.001843,0.001229,0.003071,0.001229,0.001843,0.001229,0.000614,0.000614,0.001843,NaN,NaN
2017-03,1.0,0.004395,0.003596,0.003995,0.003596,0.001598,0.001598,0.003196,0.003196,0.000799,0.003596,0.001199,0.001998,0.001199,0.001598,0.002397,0.000799,0.001199,NaN,NaN
2017-04,1.0,0.006206,0.002216,0.001773,0.002660,0.002660,0.003546,0.003103,0.003103,0.001773,0.002660,0.000887,0.000443,0.000443,0.000887,0.000887,0.001330,NaN,NaN,NaN
2017-05,1.0,0.004638,0.004638,0.002899,0.002899,0.003188,0.004058,0.001449,0.002609,0.002609,0.002609,0.003478,0.002319,0.000290,0.001739,0.002029,NaN,NaN,NaN,NaN
2017-06,1.0,0.004939,0.003951,0.004281,0.002963,0.003951,0.003622,0.002305,0.001317,0.001976,0.002963,0.003622,0.001646,0.001646,0.002305,NaN,NaN,NaN,NaN,NaN


In [22]:
# 整体复购率
repeat_rate = (rfm['frequency'] >= 2).mean()
print(f"复购率（购买≥2次用户占比）: {repeat_rate:.1%}")
# 购买间隔分布（只看复购用户，用 customer_unique_id）
repeat_users = rfm[rfm['frequency'] >= 2]['customer_unique_id']
repeat_orders = df[df['customer_unique_id'].isin(repeat_users)].sort_values(['customer_unique_id', 'order_purchase_timestamp'])
repeat_orders['prev_order'] = repeat_orders.groupby('customer_unique_id')['order_purchase_timestamp'].shift(1)
repeat_orders['days_between'] = (repeat_orders['order_purchase_timestamp'] - repeat_orders['prev_order']).dt.days
print(repeat_orders['days_between'].describe())

复购率（购买≥2次用户占比）: 3.0%
count    4412.000000
mean       55.710562
std        97.009529
min         0.000000
25%         0.000000
50%         0.000000
75%        70.000000
max       608.000000
Name: days_between, dtype: float64


In [23]:
# 各状态订单数
funnel = orders.groupby('order_status')['order_id'].nunique().sort_values(ascending=False)

# 构建关键路径漏斗：下单 → 支付 → 发货 → 签收
funnel_data = pd.DataFrame({ 'step' : [ '创建订单' , '支付成功' , '已发货' , '已签收' ], 'count' : [ len (orders),
        orders[ 'order_approved_at' ].notna(). sum (),
        orders[ 'order_delivered_carrier_date' ].notna(). sum (),
        orders[ 'order_delivered_customer_date' ].notna(). sum ()
    ]
                             })
# 计算每一步转化率
funnel_data[ 'conversion_rate' ] = funnel_data[ 'count' ] / funnel_data[ 'count' ].iloc[ 0 ]
funnel_data.to_csv( 'archive/funnel.csv' , index= False , encoding= 'utf-8-sig' )
print(funnel_data)

   step  count  conversion_rate
0  创建订单  99441         1.000000
1  支付成功  99281         0.998391
2   已发货  97658         0.982070
3   已签收  96476         0.970183


In [30]:
df['total_value'].sum()

np.float64(15418394.83)